# LiveJEPA â€” Phase 2 predictor training on Kaggle / Colab (free GPU)

Trains the small predictor head (V-JEPA 2 visual tokens â†’ SONAR caption embedding).
Everything else in LiveJEPA is frozen, so this is the **only** training in the project
and fits comfortably on a free T4/P100.

**Workflow:** upload/clone the repo â†’ install deps â†’ extract & cache features once â†’
train from the cache â†’ download `predictor_*.pt` and use it locally in the live demo.

> Select a GPU runtime first (Colab: Runtime â†’ Change runtime type â†’ T4 GPU;
> Kaggle: Settings â†’ Accelerator â†’ GPU).

In [ ]:
# --- 1. Get the repo (robust: Kaggle auto-extract, raw zip, or CWD) ---
# Kaggle auto-EXTRACTS an uploaded zip into a read-only dataset dir, so we
# detect that, copy the tree to writable /kaggle/working/repo, and cd there.
import glob, os, shutil, zipfile

DEST = '/kaggle/working/repo'

def find_repo_root():
    for marker in glob.glob('/kaggle/input/**/configs/default.yaml', recursive=True):
        root = os.path.dirname(os.path.dirname(marker))
        if os.path.isdir(os.path.join(root, 'livejepa')):
            return root
    return None

if os.path.exists('configs/default.yaml'):
    print('repo already in CWD')
else:
    zips = glob.glob('/kaggle/input/**/livejepa_kaggle.zip', recursive=True)
    root = find_repo_root()
    if zips:                       # raw zip present -> unzip it
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(DEST)
        os.chdir(DEST)
    elif root:                     # Kaggle auto-extracted -> copy read-only tree to writable dir
        if not os.path.exists(DEST):
            shutil.copytree(root, DEST)
        os.chdir(DEST)
    else:
        raise SystemExit('Repo not found. Use + Add Input to upload livejepa_kaggle.zip, then re-run.')

print('CWD:', os.getcwd())
print(sorted(os.listdir('.')))


In [ ]:
# --- 2. Install dependencies (torch is preinstalled on Kaggle/Colab) ---
!pip install -q transformers accelerate sentencepiece einops pyyaml scikit-learn scipy opencv-python decord datasets yt-dlp
import torch
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# --- 3. Configure for real models on GPU -------------------------------
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['project']['use_stubs'] = False
cfg['project']['device'] = 'cuda'
# fairseq2 also works on Linux, but the transformers backend keeps the
# embedding space identical to the local (Windows) setup.
cfg['encoders']['sonar']['backend'] = 'transformers'
yaml.safe_dump(cfg, open('configs/colab.yaml', 'w'))
print('wrote configs/colab.yaml')

## 4. Training data â€” COIN (recommended, class-diverse)

We use **COIN** (180 procedural-task classes, step-labeled) via the repo's tested
end-to-end pipeline â€” no field-mapping needed:

1. `scripts/download_coin_subset.py --per-class N` samples N videos from **every**
   class (broad caption vocabulary â€” the lever for coherent held-out captions).
2. `scripts/build_coin_cache.py` extracts `(V-JEPA 2 tokens, SONAR target)` per
   annotated step, with a **leakage-free** class-based train/val split.
3. `livejepa/train.py` consumes the cache unchanged.

> Action100M-preview is an alternative (closest to the paper) but needs manual
> field mapping. COIN is wired and runs today, so we default to it.

In [ ]:
# --- 4. Download a class-diverse COIN subset (yt-dlp; skips dead videos) ---
import os, urllib.request
from livejepa.data.coin import COIN_ANNOTATIONS_URL
os.makedirs('eval_data', exist_ok=True)
if not os.path.exists('eval_data/COIN.json'):
    print('fetching COIN.json ...')
    urllib.request.urlretrieve(COIN_ANNOTATIONS_URL, 'eval_data/COIN.json')

# ~3 videos from each of the 180 classes (raise --per-class for more data / more
# GPU-hours). Some YouTube videos are private/deleted â€” the script just skips them.
!python scripts/download_coin_subset.py --per-class 3 --coin-json eval_data/COIN.json --out-dir eval_data/coin_videos

In [ ]:
# --- 5. Extract & cache features (run ONCE; ~1-2 s/clip on a T4) --------
# V-JEPA 2 tokens + SONAR target per step -> data_cache_coin2 / _val (leakage-free
# class split). Downloads V-JEPA 2 (~1.2 GB) + SONAR ports (~3 GB) on first run.
# TIP: save data_cache_coin2{,_val} as a Kaggle Dataset afterward so reruns skip
# straight to training (extraction is the slow part).
!python scripts/build_coin_cache.py --config configs/colab.yaml --eval-root eval_data \
        --train-dir data_cache_coin2 --val-dir data_cache_coin2_val --max-steps-per-video 20
import torch; torch.cuda.empty_cache()

In [ ]:
# --- 6. Train the predictor from the cache ------------------------------
!python -m livejepa.train --config configs/colab.yaml \
        --data-root data_cache_coin2 --ckpt-dir checkpoints_kaggle --max-steps 20000 --batch-size 32

In [ ]:
# --- 7. Gate check + package the checkpoint to take home ---------------
# Does it generalize? cos(pred,target) on the held-out class-diverse val + captions.
# Gate: cos >= ~0.7 and mostly-correct decoded captions.
!python scripts/eval_predictor.py --ckpt checkpoints_kaggle/predictor_20000.pt --val data_cache_coin2_val --decode

!ls -lh checkpoints_kaggle/
!zip -j predictor_ckpt.zip checkpoints_kaggle/predictor_*.pt configs/colab.yaml
# Colab: files.download; Kaggle: grab it from the output panel
try:
    from google.colab import files
    files.download('predictor_ckpt.zip')
except ImportError:
    print('predictor_ckpt.zip ready in the working directory')

## Back on the local machine

Download `predictor_ckpt.zip`, unzip the `.pt` into `checkpoints/`, then:

```bash
# live demo (event-driven captions)
python -m livejepa.demo --config configs/default.yaml --source 0 --ckpt checkpoints/predictor_20000.pt

# full-system Pareto on HELD-OUT COIN videos (the real Phase-3 curve)
python -m livejepa.run_eval --dataset coin --mode predictor --coin-split val \
       --ckpt checkpoints/predictor_20000.pt --max-videos 10 --max-seconds 180
```

**Free-tier survival tips:** Colab disconnects ~idle 90 min / max 12 h â€” checkpoint
every `ckpt_every` steps so you lose little; Kaggle gives ~30 GPU-h/week and persists
`/kaggle/working`. **Feature extraction is the slow part** â€” save
`data_cache_coin2{,_val}` as a Kaggle Dataset so reruns skip straight to training.